# Puntuaciones y máscaras

**Capítulo 5 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_attention-mechanisms-and-transformers/attention-scoring-functions.ipynb` · [Lección original](https://d2l.ai/chapter_attention-mechanisms-and-transformers/attention-scoring-functions.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Funciones de puntuación de la atención
<a id="sec_attention-scoring-functions"></a>

En [Referencia sec_attention-pooling](https://d2l.ai/chapter_attention-mechanisms-and-transformers/attention-pooling.html#sec-attention-pooling), utilizamos una serie de diferentes núcleos basados en distancia, incluyendo un núcleo gaussiano para modelar interacciones entre consultas y claves. Resulta que las funciones de distancia son ligeramente más costosas de calcular que los productos de punto. Como tal, con la operación softmax para asegurar pesos de atención no negativos, gran parte del trabajo ha entrado en *funciones de puntuación de atención* $a$ en [Referencia eq_softmax_attention](https://d2l.ai/#eq-softmax-attention) y [Referencia fig_attention_output](https://d2l.ai/chapter_attention-mechanisms-and-transformers/attention-scoring-functions.html#fig-attention-output) que son más simples de calcular.

![La atención calcula una media ponderada de valores; los pesos resultan de la función de puntuación $\mathit{a}$ y softmax.](../recursos/originales/attention-output.svg)
<a id="fig_attention_output"></a>


In [ ]:
import math
import torch
from torch import nn
from laboratorio import d2l

## Atención por producto escalar

Repasemos la función de atención (sin exponenciación) del núcleo gaussiano por un momento:

$$
a(\mathbf{q}, \mathbf{k}_i) = -\frac{1}{2} \|\mathbf{q} - \mathbf{k}_i\|^2  = \mathbf{q}^\top \mathbf{k}_i -\frac{1}{2} \|\mathbf{k}_i\|^2  -\frac{1}{2} \|\mathbf{q}\|^2.
$$

En primer lugar, tenga en cuenta que el término final depende sólo de $\mathbf{q}$. Como tal es idéntico para todos los pares $(\mathbf{q}, \mathbf{k}_i)$. La normalización de los pesos de atención a $1$, como se hace en [Referencia eq_softmax_attention](https://d2l.ai/#eq-softmax-attention), asegura que este término desaparece por completo. En segundo lugar, tenga en cuenta que tanto la normalización por lotes (BatchNorm) y capas (que se discutirá más adelante) conducen a activaciones que han consolidado, y a menudo constantes, las normas $\|\mathbf{k}_i\|$. Este es el caso, por ejemplo, cuando las claves $\mathbf{k}_i$ fueron generadas por una norma de capa. Como tal, podemos dejarlo de la definición de $a$ sin ningún cambio importante en el resultado.

Por último, tenemos que mantener el orden de magnitud de los argumentos en la función exponencial bajo control. Supongamos que todos los elementos de la consulta $\mathbf{q} \in \mathbb{R}^d$ y la clave $\mathbf{k}_i \in \mathbb{R}^d$ son variables aleatorias independientes e idénticas dibujadas con cero media y varianza de unidades. El producto punto entre ambos vectores tiene media cero y una varianza de $d$. Para asegurar que la varianza del producto punto sigue siendo $1$ independientemente de la longitud del vector, utilizamos la función *scaled punto atención producto* puntuación. Es decir, reescalamos el producto punto por $1/\sqrt{d}$. Así llegamos a la primera función de atención comúnmente utilizada que se utiliza, por ejemplo, en Transformers [Vaswani.Shazeer.Parmar.ea.2017](https://d2l.ai/chapter_references/zreferences.html):

$$ a(\mathbf{q}, \mathbf{k}_i) = \mathbf{q}^\top \mathbf{k}_i / \sqrt{d}.$$

:eqlabel:`eq_dot_product_attention`

Tenga en cuenta que los pesos de atención $\alpha$ todavía necesitan normalización. Podemos simplificar esto más a través de [Referencia eq_softmax_attention](https://d2l.ai/#eq-softmax-attention) mediante el uso de la operación softmax:

$$\alpha(\mathbf{q}, \mathbf{k}_i) = \mathrm{softmax}(a(\mathbf{q}, \mathbf{k}_i)) = \frac{\exp(\mathbf{q}^\top \mathbf{k}_i / \sqrt{d})}{\sum_{j=1} \exp(\mathbf{q}^\top \mathbf{k}_j / \sqrt{d})}.$$

:eqlabel:`eq_attn-scoring-alpha`

Resulta que todos los mecanismos de atención popular utilizan el softmax, por lo que nos limitaremos a eso en el resto de este capítulo.

## Funciones de conveniencia
Necesitamos algunas funciones para que el mecanismo de atención sea eficiente de implementar. Esto incluye herramientas para lidiar con cadenas de longitudes variables (común para el procesamiento de lenguaje natural) y herramientas para una evaluación eficiente en minibatches (multiplicación de matriz de lotes).

### Operación Softmax enmascarada

Una de las aplicaciones más populares del mecanismo de atención es la secuencia de modelos. Por lo tanto necesitamos ser capaces de lidiar con secuencias de diferentes longitudes. En algunos casos, tales secuencias pueden terminar en el mismo minibatch, necesitando relleno con tokens ficticios para secuencias más cortas (ver [Referencia sec_machine_translation](https://d2l.ai/chapter_recurrent-modern/machine-translation-and-dataset.html#sec-machine-translation) para un ejemplo). Estos tokens especiales no tienen significado. Por ejemplo, supongamos que tenemos las siguientes tres frases:

``` Dive  into  Deep    Learning Learn to    code    <blank> Hello world <blank> <blank> ```

Dado que no queremos espacios en blanco en nuestro modelo de atención, simplemente necesitamos limitar $\sum_{i=1}^n \alpha(\mathbf{q}, \mathbf{k}_i) \mathbf{v}_i$ a $\sum_{i=1}^l \alpha(\mathbf{q}, \mathbf{k}_i) \mathbf{v}_i$ por el tiempo que sea, $l \leq n$, la frase real es. Dado que es un problema tan común, tiene un nombre: la operación *enmascarada softmax*.

Vamos a implementarlo. En realidad, la implementación engaña siempre tan ligeramente al establecer los valores de $\mathbf{v}_i$, para $i > l$, a cero. Por otra parte, establece los pesos de atención a un gran número negativo, como $-10^{6}$, con el fin de hacer su contribución a los gradientes y los valores desaparecen en la práctica. Esto se hace ya que los núcleos de álgebra lineal y los operadores están muy optimizados para GPUs y es más rápido ser un poco derrochador en la computación en lugar de tener código con condicional (si entonces) declaraciones.


In [ ]:
def masked_softmax(X, valid_lens):  #@save
    """Realizar operación softmax enmascarando elementos en el último eje."""
    # X: Tensor 3D, valid_lens: Tensor 1D o 2D
    def _sequence_mask(X, valid_len, value=0):
        maxlen = X.size(1)
        mask = torch.arange((maxlen), dtype=torch.float32,
                            device=X.device)[None, :] < valid_len[:, None]
        X[~mask] = value
        return X

    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:
            valid_lens = torch.repeat_interleave(valid_lens, shape[1])
        else:
            valid_lens = valid_lens.reshape(-1)
        # En el último eje, reemplazar los elementos enmascarados con un negativo muy grande
        # valor, cuya exponenciación sale 0
        X = _sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6)
        return nn.functional.softmax(X.reshape(shape), dim=-1)

Para **ilustrar cómo funciona esta función**, considere un minibatch de dos ejemplos de tamaño $2 \times 4$, donde sus longitudes válidas son $2$ y $3$, respectivamente. Como resultado de la operación softmax enmascarada, los valores más allá de las longitudes válidas para cada par de vectores están todos enmascarados como cero.


In [ ]:
masked_softmax(torch.rand(2, 2, 4), torch.tensor([2, 3]))

Si necesitamos más control de grano fino para especificar la longitud válida para cada uno de los dos vectores de cada ejemplo, simplemente usamos un tensor bidimensional de longitudes válidas.


In [ ]:
masked_softmax(torch.rand(2, 2, 4), torch.tensor([[1, 3], [2, 4]]))

### Multiplicación de la matriz por lotes
<a id="subsec_batch_dot"></a>

Otra operación de uso común es multiplicar los lotes de matrices entre sí. Esto es útil cuando tenemos minibatches de consultas, claves y valores. Más específicamente, asumir que

$$\mathbf{Q} = [\mathbf{Q}_1, \mathbf{Q}_2, \ldots, \mathbf{Q}_n]  \in \mathbb{R}^{n \times a \times b}, \\
    \mathbf{K} = [\mathbf{K}_1, \mathbf{K}_2, \ldots, \mathbf{K}_n]  \in \mathbb{R}^{n \times b \times c}.
$$

A continuación, la multiplicación de la matriz por lotes (BMM) calcula el producto elementowise

$$\textrm{BMM}(\mathbf{Q}, \mathbf{K}) = [\mathbf{Q}_1 \mathbf{K}_1, \mathbf{Q}_2 \mathbf{K}_2, \ldots, \mathbf{Q}_n \mathbf{K}_n] \in \mathbb{R}^{n \times a \times c}.$$

:eqlabel:`eq_batch-matrix-mul`

Veamos esto en acción en un biblioteca de aprendizaje profundo.


In [ ]:
Q = torch.ones((2, 3, 4))
K = torch.ones((2, 4, 6))
d2l.check_shape(torch.bmm(Q, K), (2, 3, 6))

## Atención por producto escalar escalado

Volvamos a la atención del producto punto introducido en [Referencia eq_dot_product_attention](https://d2l.ai/#eq-dot-product-attention). En general, requiere que tanto la consulta como la clave tienen la misma longitud vectorial, digamos $d$, aunque esto se puede abordar fácilmente reemplazando $\mathbf{q}^\top \mathbf{k}$ con $\mathbf{q}^\top \mathbf{M} \mathbf{k}$ donde $\mathbf{M}$ es una matriz convenientemente elegida para traducir entre ambos espacios. Por ahora asumir que las dimensiones coinciden.

En la práctica, a menudo pensamos en minibatches para la eficiencia, como la atención informática para $n$ consultas y $m$ pares de valor clave, donde las consultas y claves son de longitud $d$ y los valores son de longitud $v$. La atención de producto punto escalado de consultas $\mathbf Q\in\mathbb R^{n\times d}$, claves $\mathbf K\in\mathbb R^{m\times d}$ y valores $\mathbf V\in\mathbb R^{m\times v}$ por lo tanto se puede escribir como

$$ \mathrm{softmax}\left(\frac{\mathbf Q \mathbf K^\top }{\sqrt{d}}\right) \mathbf V \in \mathbb{R}^{n\times v}.$$

:eqlabel:`eq_softmax_QK_V`

Tenga en cuenta que al aplicar esto a un minibatch, necesitamos la multiplicación de la matriz por lotes introducida en [Referencia eq_batch-matrix-mul](https://d2l.ai/#eq-batch-matrix-mul). En la siguiente implementación de la atención por producto escalar escalado, utilizamos la dropout para regularización del modelo.


In [ ]:
class DotProductAttention(nn.Module):  #@save
    """Atención al producto escalar escalado."""
    def __init__(self, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    # Forma de las consultas: (batch_size, no. de consultas, d)
    # Forma de las teclas: (batch_size, número de pares de valores de teclas, d)
    # Forma de los valores: (batch_size, no. de pares de valores clave, dimensión de valor)
    # Forma de valid_lens: (batch_size,) o (batch_size, no. de consultas)
    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]
        # Intercambia las dos últimas dimensiones de las teclas con las teclas.transpose(1, 2)
        scores = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)

### Nota docente de Hespérides

Escribe qué información puede ver cada posición. Una máscara causal impide consultar el futuro; una máscara de padding excluye posiciones que no son datos. Comprueba que cada fila de atención suma uno antes de aplicar dropout. Los mapas de atención describen mezclas de valores, pero por sí solos no prueban una explicación causal del modelo.

Vínculo con los apuntes: sesión 5, «Puntuaciones y máscaras».


Para ilustrar cómo funciona la clase `DotProductAttention`**, utilizamos las mismas claves, valores y longitudes válidas del ejemplo de juguete anterior para la atención aditiva. A los efectos de nuestro ejemplo, suponemos que tenemos un tamaño de minibatch de $2$, un total de claves y valores $10$, y que la dimensión de los valores es $4$. Por último, suponemos que la longitud válida por observación es $2$ y $6$ respectivamente. Dado que esperamos que la salida sea un tensor $2 \times 1 \times 4$, es decir, una fila por ejemplo del minibatch.


In [ ]:
queries = torch.normal(0, 1, (2, 1, 2))
keys = torch.normal(0, 1, (2, 10, 2))
values = torch.normal(0, 1, (2, 10, 4))
valid_lens = torch.tensor([2, 6])

attention = DotProductAttention(dropout=0.5)
attention.eval()
d2l.check_shape(attention(queries, keys, values, valid_lens), (2, 1, 4))

Vamos a comprobar si los pesos de la atención realmente desaparecen para cualquier cosa más allá de la segunda y sexta columna, respectivamente (debido a establecer la longitud válida a $2$ y $6$).


In [ ]:
d2l.show_heatmaps(attention.attention_weights.reshape((1, 1, 2, 10)),
                  xlabel='Claves', ylabel='Consultas')

## Atención aditiva
<a id="subsec_additive-attention"></a>

Cuando las consultas $\mathbf{q}$ y las claves $\mathbf{k}$ son vectores de diferente dimensión, podemos utilizar una matriz para abordar el desajuste a través de $\mathbf{q}^\top \mathbf{M} \mathbf{k}$, o podemos utilizar la atención aditiva como función de puntuación. Otro beneficio es que, como su nombre indica, la atención es aditiva. Esto puede conducir a un pequeño ahorro computacional. Dada una consulta $\mathbf{q} \in \mathbb{R}^q$ y una clave $\mathbf{k} \in \mathbb{R}^k$, la función *aditiva* de puntuación [Bahdanau.Cho.Bengio.2014](https://d2l.ai/chapter_references/zreferences.html) es dada por

$$a(\mathbf q, \mathbf k) = \mathbf w_v^\top \textrm{tanh}(\mathbf W_q\mathbf q + \mathbf W_k \mathbf k) \in \mathbb{R},$$

:eqlabel:`eq_additive-attn`

donde $\mathbf W_q\in\mathbb R^{h\times q}$, $\mathbf W_k\in\mathbb R^{h\times k}$ y $\mathbf w_v\in\mathbb R^{h}$ son los parámetros que pueden aprenderse. Este término se introduce en un softmax para garantizar tanto la no negatividad como la normalización. Una interpretación equivalente de [Referencia eq_additive-attn](https://d2l.ai/#eq-additive-attn) es que la consulta y la clave se concatenan y se introducen en un MLP con una sola capa oculta. Usando $\tanh$ como función de activación y términos de sesgo desactivante, implementamos la siguiente atención aditiva:


In [ ]:
class AdditiveAttention(nn.Module):  #@save
    """Atención aditiva."""
    def __init__(self, num_hiddens, dropout, **kwargs):
        super(AdditiveAttention, self).__init__(**kwargs)
        self.W_k = nn.LazyLinear(num_hiddens, bias=False)
        self.W_q = nn.LazyLinear(num_hiddens, bias=False)
        self.w_v = nn.LazyLinear(1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens):
        queries, keys = self.W_q(queries), self.W_k(keys)
        # Después de la expansión de la dimensión, forma de las consultas: (batch_size, no. de
        # consultas, 1, num_hiddens) y forma de las teclas: (batch_size, 1, no. de
        # pares key-value, num_hiddens). Sumérjalos con la transmisión
        features = queries.unsqueeze(2) + keys.unsqueeze(1)
        features = torch.tanh(features)
        # Sólo hay una salida de self.w_v, por lo que eliminamos el último
        # entrada unidimensional de la forma. Forma de las puntuaciones: (batch_size,
        # no. de consultas, no. de pares de valores clave)
        scores = self.w_v(features).squeeze(-1)
        self.attention_weights = masked_softmax(scores, valid_lens)
        # Forma de los valores: (batch_size, no. de pares clave-valor, valor
        # dimensión)
        return torch.bmm(self.dropout(self.attention_weights), values)

Veamos cómo. `AdditiveAttention` En nuestro ejemplo de juguete seleccionamos consultas, claves y valores de tamaño $(2, 1, 20)$, $(2, 10, 2)$ y $(2, 10, 4)$Esto es idéntico a nuestra elección para `DotProductAttention`, excepto que ahora las consultas son $20$De la misma manera, elegimos $(2, 6)$ como longitudes válidas para las secuencias en el minibatch.


In [ ]:
queries = torch.normal(0, 1, (2, 1, 20))

attention = AdditiveAttention(num_hiddens=8, dropout=0.1)
attention.eval()
d2l.check_shape(attention(queries, keys, values, valid_lens), (2, 1, 4))

Al revisar la función de atención vemos un comportamiento cualitativamente bastante similar al de `DotProductAttention`. Es decir, sólo los términos dentro de la longitud válida elegida $(2, 6)$ son no cero.


In [ ]:
d2l.show_heatmaps(attention.attention_weights.reshape((1, 1, 2, 10)),
                  xlabel='Claves', ylabel='Consultas')

## Resumen
En esta sección introdujimos las dos funciones clave de puntuación de la atención: producto escalar y atención aditiva. Son herramientas eficaces para agregar secuencias de longitud variable. En particular, la atención del producto escalar es el pilar de las arquitecturas modernas de Transformador. Cuando las consultas y las claves son vectores de diferentes longitudes, podemos usar la función de puntuación de atención aditiva en su lugar. Optimizar estas capas es una de las áreas clave de avance en los últimos años. Por ejemplo, [NVIDIA's Transformer Library](https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/index.html) y Megatron [shoeybi2019megatron](https://d2l.ai/chapter_references/zreferences.html) dependen crucialmente de variantes eficientes del mecanismo de atención. Nos sumergiremos en esto con bastante más detalle mientras revisamos Transformadores en secciones posteriores.

## Ejercicios
1. Implemente la atención basada en la distancia modificando el código `DotProductAttention`. Tenga en cuenta que solo necesita las normas cuadradas de las claves $\|\mathbf{k}_i\|^2$ para una implementación eficiente.
1. Modificar la atención del producto punto para permitir consultas y claves de diferentes dimensiones mediante el empleo de una matriz para ajustar las dimensiones.
1. ¿Cómo se escala el costo computacional con la dimensión de las claves, consultas, valores y su número? ¿Qué pasa con los requisitos de ancho de banda de la memoria?


[Debate del original](https://discuss.d2l.ai/t/1064)
